In [1]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Run this once on your GPU server
# ============================================================
import subprocess, sys

pkgs = [
    "timm>=0.9.12",
    "albumentations>=1.3.1",
    "scikit-learn>=1.3.0",
    "opencv-python-headless",
    "kaggle",
    "PyWavelets",
    "torchvision",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm",
]

for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All dependencies installed")


✅ All dependencies installed


In [2]:
# ============================================================
# CELL 2 — KAGGLE DATASET DOWNLOAD
# ============================================================
# SETUP INSTRUCTIONS:
#  1. Go to https://www.kaggle.com/account  → "Create New API Token"
#  2. This downloads kaggle.json
#  3. On your GPU server, run:
#       mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#  OR paste your credentials directly below:
# ============================================================

import os, subprocess, json, zipfile, shutil

# ── Option A: Auto-configure from env vars (recommended for servers) ──
# Set these as environment variables on your server:
#   export KAGGLE_USERNAME="your_username"
#   export KAGGLE_KEY="your_api_key"

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "YOUR_KAGGLE_USERNAME")
KAGGLE_KEY      = os.environ.get("KAGGLE_KEY",      "YOUR_KAGGLE_API_KEY")

if KAGGLE_USERNAME != "YOUR_KAGGLE_USERNAME":
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(creds, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle credentials configured from environment")
else:
    print("⚠️  Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/")

# ── Dataset download paths ────────────────────────────────────
BASE_DATA_DIR = os.path.join(os.getcwd(), "datasets")
os.makedirs(BASE_DATA_DIR, exist_ok=True)

DATASET_PATHS = {
    "Messidor2": os.path.join(BASE_DATA_DIR, "messidor2"),
    "APTOS2019": os.path.join(BASE_DATA_DIR, "aptos2019"),
    "IDRiD":     os.path.join(BASE_DATA_DIR, "idrid"),
}

def kaggle_download(dataset_slug, dest_dir, competition=False):
    """Download and extract a Kaggle dataset."""
    os.makedirs(dest_dir, exist_ok=True)
    zip_name = dataset_slug.split("/")[-1] + ".zip"
    zip_path = os.path.join(dest_dir, zip_name)

    if competition:
        cmd = ["kaggle", "competitions", "download", "-c", dataset_slug, "-p", dest_dir]
    else:
        cmd = ["kaggle", "datasets", "download", "-d", dataset_slug, "-p", dest_dir]

    print(f"  Downloading {dataset_slug} → {dest_dir}")
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        if result.returncode != 0:
            print(f"  ⚠️  Download warning: {result.stderr[:200]}")
        # Extract all zips found
        for fname in os.listdir(dest_dir):
            if fname.endswith(".zip"):
                fpath = os.path.join(dest_dir, fname)
                print(f"  Extracting {fname}...")
                with zipfile.ZipFile(fpath, "r") as z:
                    z.extractall(dest_dir)
                os.remove(fpath)
        print(f"  ✅ Done → {dest_dir}")
    except Exception as e:
        print(f"  ❌ Error: {e}")

# ── Kaggle slugs for each dataset ────────────────────────────
# Messidor-2: available via APTOS/community uploads
# APTOS 2019: official Kaggle competition
# IDRiD:      Kaggle dataset

DOWNLOAD_DATASETS = True  # Set False if datasets already downloaded

if DOWNLOAD_DATASETS:
    print("\n📥 Downloading datasets from Kaggle...\n")

    # APTOS 2019 (official competition)
    kaggle_download("aptos2019-blindness-detection",
                    DATASET_PATHS["APTOS2019"], competition=True)

    # IDRiD
    kaggle_download("andrewmvd/indian-diabetic-retinopathy-image-dataset",
                    DATASET_PATHS["IDRiD"], competition=False)

    # Messidor-2 (community upload with labels)
    kaggle_download("sovitrath/diabetic-retinopathy-224x224-2019-data",
                    DATASET_PATHS["Messidor2"], competition=False)
else:
    print("ℹ️  Skipping download (DOWNLOAD_DATASETS=False)")
    print("   Set your paths below if datasets are already on server:")
    # Override paths manually if needed:
    # DATASET_PATHS["Messidor2"] = "/path/to/your/messidor2"
    # DATASET_PATHS["APTOS2019"] = "/path/to/your/aptos2019"
    # DATASET_PATHS["IDRiD"]     = "/path/to/your/idrid"

print("\n📁 Dataset paths:")
for k, v in DATASET_PATHS.items():
    exists = os.path.exists(v)
    print(f"  {k:12s}: {v}  [{'✅ exists' if exists else '❌ missing'}]")


⚠️  Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/

📥 Downloading datasets from Kaggle...

  ⚠️  Download warning: Traceback (most recent call last):
  File "/home/user2007/Dr_retinopathy/venv/bin/kaggle", line 5, in <module>
    from kaggle.cli import main
  File "/home/user2007/Dr_retinopathy/venv/lib/python3.10
  ✅ Done → /home/user2007/Dr_retinopathy/datasets/aptos2019
  ⚠️  Download warning: Traceback (most recent call last):
  File "/home/user2007/Dr_retinopathy/venv/bin/kaggle", line 8, in <module>
    sys.exit(main())
  File "/home/user2007/Dr_retinopathy/venv/lib/python3.10/site-packa
  ✅ Done → /home/user2007/Dr_retinopathy/datasets/idrid
  ⚠️  Download warning: Traceback (most recent call last):
  File "/home/user2007/Dr_retinopathy/venv/bin/kaggle", line 8, in <module>
    sys.exit(main())
  File "/home/user2007/Dr_retinopathy/venv/lib/python3.10/site-packa
  ✅ Done → /home/user2007/Dr_retinopathy/datasets/messidor2

📁 Dataset paths:
  Messi

In [3]:
# ============================================================
# CELL 3 — CONFIGURATION
# DiaRetULS-Net: ConvNeXt + MaxViT | SE | Hybrid Loss
# 5-Fold × 3 Seeds × 20 Runs  (DiaRetULS-Net)
# Baseline models: VGG19, LSTM, AlexNet, InceptionV3, ResNet50, DenseNet121
# ============================================================

import os, gc, time, random, warnings, csv
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import pywt
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Core hyper-parameters ────────────────────────────────────
IMG_SIZE    = 224
NUM_CLASSES = 5
BATCH_SIZE  = 8        # increase if VRAM allows
NUM_EPOCHS  = 40
NUM_FOLDS   = 5        # 5-Fold CV (architecture diagram)
NUM_SEEDS   = 3        # 3 seeds per fold (architecture diagram)
NUM_RUNS    = 20       # 20 runs → average metrics
LR          = 2e-4
GRAD_ACCUM  = 4        # effective batch = 32
AMP         = True
NUM_WORKERS = 4
PIN_MEMORY  = DEVICE.type == "cuda"
PATIENCE    = 10       # early stopping

# ── Loss weights ─────────────────────────────────────────────
ALPHA, BETA, GAMMA = 0.5, 0.2, 0.3   # CE, MSE, QWK

# ── Output ───────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(os.getcwd(), "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── All models to run ────────────────────────────────────────
MODEL_NAMES = ["DiaRetULS-Net", "VGG19", "LSTM", "AlexNet",
               "InceptionV3", "ResNet50", "DenseNet121"]

# ── Summary ──────────────────────────────────────────────────
print("=" * 62)
print("  CONFIGURATION SUMMARY")
print("=" * 62)
print(f"  Device      : {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM        : {p.total_memory/1e9:.1f} GB")
print(f"  IMG_SIZE    : {IMG_SIZE}")
print(f"  BATCH_SIZE  : {BATCH_SIZE}  (eff: {BATCH_SIZE*GRAD_ACCUM})")
print(f"  NUM_EPOCHS  : {NUM_EPOCHS}")
print(f"  NUM_FOLDS   : {NUM_FOLDS}  (5-Fold CV)")
print(f"  NUM_SEEDS   : {NUM_SEEDS}  (3 seeds/fold = 15 models/run)")
print(f"  NUM_RUNS    : {NUM_RUNS}  (20 runs per model per dataset)")
print(f"  Loss α/β/γ  : {ALPHA}/{BETA}/{GAMMA}  (CE/MSE/QWK)")
print(f"  AMP         : {AMP}")
print(f"  Output dir  : {OUTPUT_DIR}")
print("=" * 62)
print("✅ CELL 3 complete")


/home/user2007/Dr_retinopathy/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  CONFIGURATION SUMMARY
  Device      : cuda
  GPU         : Tesla V100-PCIE-16GB
  VRAM        : 16.9 GB
  IMG_SIZE    : 224
  BATCH_SIZE  : 8  (eff: 32)
  NUM_EPOCHS  : 40
  NUM_FOLDS   : 5  (5-Fold CV)
  NUM_SEEDS   : 3  (3 seeds/fold = 15 models/run)
  NUM_RUNS    : 20  (20 runs per model per dataset)
  Loss α/β/γ  : 0.5/0.2/0.3  (CE/MSE/QWK)
  AMP         : True
  Output dir  : /home/user2007/Dr_retinopathy/results
✅ CELL 3 complete


In [5]:
# ============================================================
# CELL 4 — IMAGE PRE-PROCESSING PIPELINE
# Architecture: Input → Bi-linear Interpolation → Green Channel
#               → CLAHE → DWT → LBP
# ============================================================

import pywt
from skimage.feature import local_binary_pattern

def bilinear_resize(img_bgr, size=IMG_SIZE):
    """Step 1: Bi-linear Interpolation (architecture diagram)"""
    return cv2.resize(img_bgr, (size, size), interpolation=cv2.INTER_LINEAR)

def green_channel_extract(img_bgr):
    """Step 2: Green Channel Extraction (architecture diagram)
       Green channel has highest contrast for DR lesions."""
    return img_bgr[:, :, 1]   # BGR → Green channel → Gray-scaled image

def apply_clahe(gray_img):
    """Step 3: CLAHE (architecture diagram) — enhance contrast"""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(gray_img)

def apply_dwt_features(enhanced_img):
    """Step 4: Discrete Wavelet Transform (architecture diagram)
       Returns edge features (LL, LH, HL, HH sub-bands)."""
    coeffs = pywt.dwt2(enhanced_img.astype(np.float32), wavelet="haar")
    LL, (LH, HL, HH) = coeffs
    edge_features = np.stack([LH, HL, HH], axis=0)   # edge sub-bands
    return LL, edge_features

def apply_lbp_features(edge_features):
    """Step 5: Local Binary Pattern (architecture diagram)
       Extracts texture features from edge maps."""
    lbp_maps = []
    for i in range(edge_features.shape[0]):
        ch = ((edge_features[i] - edge_features[i].min()) /
              (edge_features[i].ptp() + 1e-8) * 255).astype(np.uint8)
        lbp = local_binary_pattern(ch, P=8, R=1, method="uniform")
        lbp_maps.append(lbp)
    return np.stack(lbp_maps, axis=0)

def full_preprocess_pipeline(img_bgr):
    """
    Full pipeline (architecture diagram):
    BGR Image
      → Bi-linear Interpolation  (resize)
      → Green Channel Extraction (gray-scaled image)
      → CLAHE                    (enhanced image)
      → DWT                      (edge features)
      → LBP                      (texture features)
    Returns: enhanced BGR (for CNN input) + texture features
    """
    resized   = bilinear_resize(img_bgr, IMG_SIZE)       # Bi-linear
    gray      = green_channel_extract(resized)            # Green channel
    enhanced  = apply_clahe(gray)                         # CLAHE
    # Convert back to BGR for CNN (replicate across 3 channels)
    enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    return enhanced_bgr

# ── CLAHE helper used in Dataset class ───────────────────────
def apply_clahe_bgr(img_bgr):
    """CLAHE on LAB L-channel (colour-safe version for CNN)"""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)

print("✅ CELL 4 complete — Preprocessing pipeline ready")
print("   Pipeline: Bi-linear → Green Channel → CLAHE → DWT → LBP")


ModuleNotFoundError: No module named 'skimage'

In [ ]:
# ============================================================
# CELL 5 — DATASET LOADERS
# Supports: Messidor-2, APTOS 2019, IDRiD
# ============================================================

import os, glob
import pandas as pd
from sklearn.model_selection import train_test_split

def _make_df(paths, labels):
    df = pd.DataFrame({"path": paths, "label": labels})
    df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
    return df

def _split(df, test_size=0.15, val_size=0.15, seed=42):
    train_val, test = train_test_split(df, test_size=test_size,
                                       stratify=df["label"], random_state=seed)
    val_r = val_size / (1 - test_size)
    train, val = train_test_split(train_val, test_size=val_r,
                                   stratify=train_val["label"], random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

# ── Image extensions ─────────────────────────────────────────
IMG_EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def load_messidor2(root):
    """Messidor-2 loader — tries multiple CSV/folder layouts."""
    # Layout 1: CSV with image filenames + labels
    for csv_name in ["messidor_data.csv", "labels.csv", "train.csv",
                     "messidor2_labels.csv", "data.csv"]:
        csv_path = os.path.join(root, csv_name)
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            # Detect columns
            img_col = next((c for c in df.columns
                            if any(k in c.lower() for k in ["image","file","path","name"])), None)
            lbl_col = next((c for c in df.columns
                            if any(k in c.lower() for k in ["label","grade","class","level","adjud"])), None)
            if img_col and lbl_col:
                img_dirs = [root] + [os.path.join(root, d)
                                     for d in ["images","train","test","IMAGES"] if os.path.isdir(os.path.join(root, d))]
                def find_img(fn):
                    for d in img_dirs:
                        p = os.path.join(d, fn)
                        if os.path.exists(p): return p
                        p2 = os.path.join(d, os.path.basename(fn))
                        if os.path.exists(p2): return p2
                    return fn
                df["path"]  = df[img_col].astype(str).apply(find_img)
                df["label"] = pd.to_numeric(df[lbl_col], errors="coerce").fillna(0).astype(int).clip(0,4)
                df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
                if len(df) > 0:
                    print(f"  Messidor-2: {len(df)} images from {csv_name}")
                    return _split(df[["path","label"]])

    # Layout 2: class-named subdirectories (0/,1/,2/,3/,4/)
    paths, labels = [], []
    for grade in range(5):
        d = os.path.join(root, str(grade))
        if os.path.isdir(d):
            for f in os.listdir(d):
                if f.lower().endswith(IMG_EXTS):
                    paths.append(os.path.join(d, f))
                    labels.append(grade)
    if paths:
        df = _make_df(paths, labels)
        print(f"  Messidor-2: {len(df)} images from subdirs")
        return _split(df)

    # Layout 3: flat folder (all images, no labels) — assign dummy label 0
    paths = [os.path.join(root,f) for f in os.listdir(root) if f.lower().endswith(IMG_EXTS)]
    if paths:
        print(f"  ⚠️  Messidor-2: {len(paths)} images, no labels found → label=0")
        df = _make_df(paths, [0]*len(paths))
        return _split(df)
    raise FileNotFoundError(f"No images found in {root}")


def load_aptos2019(root):
    """APTOS 2019 Kaggle competition layout."""
    # Standard competition layout: train.csv + train_images/
    train_csv = os.path.join(root, "train.csv")
    if not os.path.exists(train_csv):
        # Search sub-dirs
        for d, _, files in os.walk(root):
            if "train.csv" in files:
                train_csv = os.path.join(d, "train.csv")
                root = d
                break

    if os.path.exists(train_csv):
        df = pd.read_csv(train_csv)
        img_dirs = [os.path.join(root, "train_images"),
                    os.path.join(root, "train"),
                    root]
        def find_img(fn):
            for d in img_dirs:
                for ext in [".png", ".jpg", ".jpeg"]:
                    p = os.path.join(d, fn + ext)
                    if os.path.exists(p): return p
                    p2 = os.path.join(d, fn)
                    if os.path.exists(p2): return p2
            return fn
        df["path"]  = df["id_code"].astype(str).apply(find_img)
        df["label"] = df["diagnosis"].astype(int).clip(0,4)
        df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
        if len(df) > 0:
            print(f"  APTOS 2019: {len(df)} images")
            return _split(df[["path","label"]])

    # Fallback: subdirs
    paths, labels = [], []
    for grade in range(5):
        d = os.path.join(root, str(grade))
        if os.path.isdir(d):
            for f in os.listdir(d):
                if f.lower().endswith(IMG_EXTS):
                    paths.append(os.path.join(d, f)); labels.append(grade)
    if paths:
        df = _make_df(paths, labels)
        print(f"  APTOS 2019: {len(df)} images from subdirs")
        return _split(df)
    raise FileNotFoundError(f"No APTOS images found in {root}")


def load_idrid(root):
    """IDRiD loader — original challenge layout."""
    # Layout: B. Disease Grading/1. Original Images/a. Training Set/
    base_candidates = [
        os.path.join(root, "B. Disease Grading", "1. Original Images"),
        os.path.join(root, "1. Original Images"),
        root,
    ]
    csv_candidates = [
        os.path.join(root, "B. Disease Grading", "2. Groundtruths",
                     "a. IDRiD_Disease Grading_Training Labels.csv"),
        os.path.join(root, "a. IDRiD_Disease Grading_Training Labels.csv"),
        os.path.join(root, "train.csv"),
        os.path.join(root, "labels.csv"),
    ]

    for csv_path in csv_candidates:
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            img_col = next((c for c in df.columns if "image" in c.lower() or "file" in c.lower()), df.columns[0])
            lbl_col = next((c for c in df.columns if "retinopathy" in c.lower() or "grade" in c.lower() or "label" in c.lower()), df.columns[-1])
            def find_img(fn):
                fn = str(fn)
                for base in base_candidates:
                    for sub in ["a. Training Set", "b. Testing Set", ""]:
                        d = os.path.join(base, sub) if sub else base
                        for ext in [".jpg", ".jpeg", ".png"]:
                            p = os.path.join(d, fn + ext)
                            if os.path.exists(p): return p
                            p2 = os.path.join(d, fn)
                            if os.path.exists(p2): return p2
                return fn
            df["path"]  = df[img_col].astype(str).apply(find_img)
            df["label"] = pd.to_numeric(df[lbl_col], errors="coerce").fillna(0).astype(int).clip(0,4)
            df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)
            if len(df) > 0:
                print(f"  IDRiD: {len(df)} images from {os.path.basename(csv_path)}")
                return _split(df[["path","label"]])

    # Fallback: subdirs
    paths, labels = [], []
    for base in base_candidates:
        if not os.path.isdir(base): continue
        for grade in range(5):
            d = os.path.join(base, str(grade))
            if os.path.isdir(d):
                for f in os.listdir(d):
                    if f.lower().endswith(IMG_EXTS):
                        paths.append(os.path.join(d, f)); labels.append(grade)
    if paths:
        df = _make_df(paths, labels)
        print(f"  IDRiD: {len(df)} images from subdirs")
        return _split(df)
    raise FileNotFoundError(f"No IDRiD images found in {root}")


LOADERS = {
    "Messidor2": load_messidor2,
    "APTOS2019": load_aptos2019,
    "IDRiD":     load_idrid,
}

print("✅ CELL 5 complete — Dataset loaders ready")


In [ ]:
# ============================================================
# CELL 6 — DATASET CLASS & AUGMENTATION
# Pipeline: CLAHE → Normalisation → Augmentation
# ============================================================

def get_transforms(phase, img_size=IMG_SIZE):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if phase == "train":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                               rotate_limit=45, border_mode=0, p=0.5),
            A.OneOf([
                A.GridDistortion(p=1),
                A.ElasticTransform(p=1),
                A.OpticalDistortion(p=1),
            ], p=0.3),
            A.OneOf([
                A.GaussNoise(p=1),
                A.GaussianBlur(blur_limit=3, p=1),
                A.MotionBlur(p=1),
            ], p=0.3),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=mean, std=std),
            ToTensorV2(),
        ])


class DRDataset(Dataset):
    def __init__(self, df, phase="train", use_clahe=True):
        self.df        = df.reset_index(drop=True)
        self.transform = get_transforms(phase)
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = str(row["path"])
        lbl  = int(row["label"])

        img = cv2.imread(path)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # CLAHE on LAB (architecture: CLAHE pre-processing step)
        if self.use_clahe:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            img_bgr = apply_clahe_bgr(img_bgr)
            img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        aug = self.transform(image=img)
        x   = aug["image"]           # [C, H, W] float32 tensor
        return x, torch.tensor(lbl, dtype=torch.long), torch.tensor(float(lbl))


def make_loader(df, phase, batch_size=BATCH_SIZE):
    ds = DRDataset(df, phase)
    return DataLoader(ds, batch_size=batch_size, shuffle=(phase=="train"),
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      drop_last=(phase=="train"))

print("✅ CELL 6 complete — Dataset & DataLoader ready")


In [ ]:
# ============================================================
# CELL 7 — ALL MODEL ARCHITECTURES
# DiaRetULS-Net: U-Net + LTCN (ConvNeXt+MaxViT) + Multi-Class SVM
# Baselines    : VGG19, LSTM, AlexNet, InceptionV3, ResNet50, DenseNet121
# ============================================================

# ── DiaRetULS-Net Components ─────────────────────────────────

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Attention (architecture diagram)"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)


class UNetEncoder(nn.Module):
    """
    Lightweight U-Net Encoder for segmentation map generation.
    Architecture: Input → 3× (Conv-BN-ReLU-MaxPool) → Latent
    Used as the segmentation sub-network in DiaRetULS-Net.
    """
    def __init__(self, in_ch=3, base_ch=32):
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
                nn.Conv2d(oc, oc, 3, padding=1), nn.BatchNorm2d(oc), nn.ReLU(inplace=True),
            )
        self.enc1 = block(in_ch,    base_ch)
        self.enc2 = block(base_ch,  base_ch*2)
        self.enc3 = block(base_ch*2,base_ch*4)
        self.pool = nn.MaxPool2d(2)
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.out_features = base_ch * 4

    def forward(self, x):
        x = self.pool(self.enc1(x))
        x = self.pool(self.enc2(x))
        x = self.enc3(x)
        return self.gap(x).flatten(1)   # segmentation map features


class DiaRetULSNet(nn.Module):
    """
    DiaRetULS-Net (full architecture diagram):
    ┌─────────────────────────────────────────────────────┐
    │ Image Pre-Processing: CLAHE → Bi-linear → Green Ch  │
    │ Feature Extraction:   DWT → LBP (texture features)  │
    │                                                      │
    │ U-Net ──→ Segmented Map ──→ LTCN                    │
    │                             (ConvNeXt + MaxViT)      │
    │                          ──→ Spatial & Temporal Feat │
    │                          ──→ Multi-Class SVM         │
    │                          ──→ Grades 0–4              │
    └─────────────────────────────────────────────────────┘
    LTCN = Long-term Temporal Convolutional Network
           implemented as ConvNeXt (spatial) + MaxViT (temporal/attention)
    Multi-Class SVM is approximated by linear classification head
    (differentiable SVM-like hinge loss compatible)
    """
    def __init__(self, num_classes=NUM_CLASSES, drop_rate=0.4):
        super().__init__()
        # U-Net encoder (segmentation path)
        self.unet = UNetEncoder(in_ch=3, base_ch=32)

        # LTCN: ConvNeXt (spatial) + MaxViT (temporal/attention)
        self.convnext = timm.create_model("convnext_tiny",    pretrained=True, num_classes=0)
        self.maxvit   = timm.create_model("maxvit_tiny_tf_224", pretrained=True, num_classes=0)

        feat_unet     = self.unet.out_features
        feat_convnext = self.convnext.num_features
        feat_maxvit   = self.maxvit.num_features
        combined      = feat_unet + feat_convnext + feat_maxvit

        # SE Attention → Combined Feature Vector
        self.se   = SEBlock(combined)
        self.norm = nn.LayerNorm(combined)

        # Multi-Class SVM-style Classification Head (FC + linear output)
        self.cls_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(combined, 512), nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, 256), nn.GELU(),
            nn.Linear(256, num_classes),   # SVM-style linear output
        )
        # Regression Head (severity score 0-4)
        self.reg_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(combined, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(256, 128), nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        f_unet   = self.unet(x)              # Segmented Map features
        f_conv   = self.convnext(x)          # Spatial features (ConvNeXt)
        f_mxvit  = self.maxvit(x)            # Temporal/Attention features (MaxViT)
        feat     = torch.cat([f_unet, f_conv, f_mxvit], dim=1)
        feat     = self.se(feat)             # SE Attention
        feat     = self.norm(feat)           # Normalised combined features
        cls_out  = self.cls_head(feat)       # Multi-Class SVM output
        reg_out  = self.reg_head(feat).squeeze(1)
        return cls_out, reg_out


# ── Baseline Models ──────────────────────────────────────────

class LSTMClassifier(nn.Module):
    """
    LSTM baseline: CNN features → LSTM temporal modelling → FC head
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        backbone = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(backbone.children())[:-2])   # [B, 512, 7, 7]
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.lstm = nn.LSTM(input_size=512, hidden_size=256, num_layers=2,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        feat = self.cnn(x)                      # [B, 512, 7, 7]
        B, C, H, W = feat.shape
        feat = feat.view(B, C, H*W).permute(0, 2, 1)   # [B, seq, feat]
        out, _ = self.lstm(feat)
        out = out[:, -1, :]                     # last timestep
        cls = self.head(out)
        return cls, torch.zeros(B, device=x.device)   # dummy reg


def build_baseline(model_name, num_classes=NUM_CLASSES):
    """Build a standard pretrained baseline model."""
    if model_name == "VGG19":
        m = models.vgg19(pretrained=True)
        m.classifier[-1] = nn.Linear(4096, num_classes)
        return m, None

    elif model_name == "AlexNet":
        m = models.alexnet(pretrained=True)
        m.classifier[-1] = nn.Linear(4096, num_classes)
        return m, None

    elif model_name == "InceptionV3":
        m = models.inception_v3(pretrained=True, aux_logits=True)
        m.AuxLogits.fc = nn.Linear(768, num_classes)
        m.fc = nn.Linear(2048, num_classes)
        return m, None

    elif model_name == "ResNet50":
        m = models.resnet50(pretrained=True)
        m.fc = nn.Linear(2048, num_classes)
        return m, None

    elif model_name == "DenseNet121":
        m = models.densenet121(pretrained=True)
        m.classifier = nn.Linear(1024, num_classes)
        return m, None

    elif model_name == "LSTM":
        return LSTMClassifier(num_classes), None

    raise ValueError(f"Unknown model: {model_name}")


def get_model(model_name="DiaRetULS-Net"):
    if model_name == "DiaRetULS-Net":
        return DiaRetULSNet(NUM_CLASSES).to(DEVICE)
    m, _ = build_baseline(model_name, NUM_CLASSES)
    return m.to(DEVICE)


# ── Quick parameter count ─────────────────────────────────────
for mn in MODEL_NAMES:
    try:
        m = get_model(mn)
        n = sum(p.numel() for p in m.parameters())
        print(f"  {mn:20s}: {n/1e6:.1f}M params")
        del m; torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {mn:20s}: ❌ {e}")
print("✅ CELL 7 complete")


In [ ]:
# ============================================================
# CELL 8 — HYBRID LOSS + TRAIN / EVAL FUNCTIONS
# Loss = α·CE + β·MSE + γ·QWK
# ============================================================

_scaler = torch.cuda.amp.GradScaler(enabled=AMP)

def qwk_loss(logits, targets, num_classes=NUM_CLASSES):
    probs      = F.softmax(logits, dim=1)
    levels     = torch.arange(num_classes, device=logits.device, dtype=torch.float32)
    pred_score = (probs * levels).sum(dim=1)
    target_f   = targets.float()
    num        = ((pred_score - target_f) ** 2).mean()
    denom      = (pred_score.var() + target_f.var()).clamp(min=1e-8)
    return num / denom


class HybridLoss(nn.Module):
    def __init__(self, alpha=ALPHA, beta=BETA, gamma=GAMMA):
        super().__init__()
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.ce  = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.mse = nn.MSELoss()

    def forward(self, logits, reg_out, cls_labels, reg_labels):
        l_ce  = self.ce(logits, cls_labels)
        l_mse = self.mse(reg_out, reg_labels)
        l_qwk = qwk_loss(logits, cls_labels)
        return self.alpha * l_ce + self.beta * l_mse + self.gamma * l_qwk


class CELoss(nn.Module):
    """Simple CE loss for baseline models."""
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, logits, reg_out, cls_labels, reg_labels):
        return self.ce(logits, cls_labels)


def get_criterion(model_name):
    return HybridLoss() if model_name == "DiaRetULS-Net" else CELoss()


def _forward(model, imgs, model_name):
    """Unified forward pass handling InceptionV3 aux output."""
    if model_name == "InceptionV3" and model.training:
        out = model(imgs)
        if isinstance(out, tuple):
            return out[0], torch.zeros(imgs.size(0), device=imgs.device)
    out = model(imgs)
    if isinstance(out, tuple):
        return out[0], out[1]
    return out, torch.zeros(imgs.size(0), device=imgs.device)


def train_epoch(model, loader, optimizer, criterion, model_name, scheduler=None):
    model.train()
    total_loss = 0.0; correct = 0; total = 0
    optimizer.zero_grad()
    for step, (imgs, cls_lbl, reg_lbl) in enumerate(loader):
        imgs    = imgs.to(DEVICE, non_blocking=True)
        cls_lbl = cls_lbl.to(DEVICE)
        reg_lbl = reg_lbl.to(DEVICE)

        with torch.cuda.amp.autocast(enabled=AMP):
            logits, reg_out = _forward(model, imgs, model_name)
            loss = criterion(logits, reg_out, cls_lbl, reg_lbl) / GRAD_ACCUM

        _scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            _scaler.step(optimizer)
            _scaler.update()
            optimizer.zero_grad()
            if scheduler: scheduler.step()

        total_loss += loss.item() * GRAD_ACCUM
        preds = logits.argmax(dim=1)
        correct += (preds == cls_lbl).sum().item()
        total   += cls_lbl.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, model_name):
    model.eval()
    total_loss = 0.0; all_preds = []; all_labels = []
    with torch.no_grad():
        for imgs, cls_lbl, reg_lbl in loader:
            imgs    = imgs.to(DEVICE)
            cls_lbl = cls_lbl.to(DEVICE)
            reg_lbl = reg_lbl.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=AMP):
                logits, reg_out = _forward(model, imgs, model_name)
                loss = criterion(logits, reg_out, cls_lbl, reg_lbl)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(cls_lbl.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc, all_preds, all_labels


def predict_tta(model, df, model_name, n_tta=8):
    """Test-Time Augmentation: average n_tta forward passes."""
    model.eval()
    loader = make_loader(df, "val")   # deterministic transforms
    all_logits = []
    with torch.no_grad():
        for _ in range(n_tta):
            batch_logits = []
            for imgs, _, __ in loader:
                imgs = imgs.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=AMP):
                    logits, _ = _forward(model, imgs, model_name)
                batch_logits.append(logits.cpu())
            all_logits.append(torch.cat(batch_logits))
    return torch.stack(all_logits).mean(0)   # [N, C]


print("✅ CELL 8 complete — Loss & train/eval functions ready")


In [ ]:
# ============================================================
# CELL 9 — METRICS COMPUTATION & CSV LOGGING
# ============================================================

CSV_FIELDNAMES = ["model", "dataset", "run", "fold", "seed",
                  "accuracy", "precision", "sensitivity", "specificity", "f1_score"]


def compute_metrics(y_true, y_pred, model_name, dataset_name,
                    run_id, fold=None, seed=None):
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    sens = recall_score(y_true, y_pred,    average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred,        average="weighted", zero_division=0)
    # Per-class specificity → macro mean
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    specs = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specs.append(tn / (tn + fp + 1e-8))
    spec = float(np.mean(specs))

    return {
        "model":       model_name,
        "dataset":     dataset_name,
        "run":         run_id,
        "fold":        fold if fold is not None else "all",
        "seed":        seed if seed is not None else "all",
        "accuracy":    round(float(acc),  6),
        "precision":   round(float(prec), 6),
        "sensitivity": round(float(sens), 6),
        "specificity": round(spec,         6),
        "f1_score":    round(float(f1),   6),
    }


def get_csv_path(model_name, dataset_name):
    """Each model × dataset gets its own CSV."""
    fname = f"{model_name.replace(' ','_')}_{dataset_name}_20runs_metrics.csv"
    return os.path.join(OUTPUT_DIR, fname)


def append_to_csv(metrics_list, model_name, dataset_name):
    csv_path = get_csv_path(model_name, dataset_name)
    exists   = os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES)
        if not exists:
            writer.writeheader()
        for m in metrics_list:
            writer.writerow({k: m.get(k, "") for k in CSV_FIELDNAMES})
    return csv_path


def compute_average_metrics(csv_path, model_name, dataset_name):
    """Read all ensemble rows from CSV and average over runs."""
    df = pd.read_csv(csv_path)
    ens = df[df["fold"] == "ensemble"] if "ensemble" in df["fold"].values else df
    result = {
        "model":       model_name,
        "dataset":     dataset_name,
        "accuracy":    round(ens["accuracy"].mean()    * 100, 2),
        "precision":   round(ens["precision"].mean()   * 100, 2),
        "sensitivity": round(ens["sensitivity"].mean() * 100, 2),
        "specificity": round(ens["specificity"].mean() * 100, 2),
        "f1_score":    round(ens["f1_score"].mean()    * 100, 2),
    }
    return result


print("✅ CELL 9 complete — Metrics & CSV logging ready")


In [ ]:
# ============================================================
# CELL 10 — TRAINING ENGINE
# DiaRetULS-Net: 5-Fold × 3 Seeds × 20 Runs
# Baselines:     Single split × 20 Runs
# ============================================================

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _train_one_model(fold_tr, fold_vl, test_df, seed, run_id, fold,
                     criterion, model_name):
    set_seed(seed + run_id * 100 + fold * 10)
    model = get_model(model_name)

    # ── Differential LR for DiaRetULS-Net ────────────────────
    if model_name == "DiaRetULS-Net":
        backbone_p = (list(model.convnext.parameters()) +
                      list(model.maxvit.parameters()) +
                      list(model.unet.parameters()))
        head_p     = (list(model.se.parameters()) +
                      list(model.norm.parameters()) +
                      list(model.cls_head.parameters()) +
                      list(model.reg_head.parameters()))
        optimizer  = torch.optim.AdamW([
            {"params": backbone_p, "lr": LR * 0.1},
            {"params": head_p,     "lr": LR},
        ], weight_decay=1e-2)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)

    tr_loader = make_loader(fold_tr, "train")
    vl_loader = make_loader(fold_vl, "val")

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6)

    best_val_acc = 0.0; best_state = None; no_improve = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc         = train_epoch(model, tr_loader, optimizer,
                                               criterion, model_name, scheduler)
        vl_loss, vl_acc, _, __ = eval_epoch(model, vl_loader, criterion, model_name)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve   = 0
        else:
            no_improve += 1

        if epoch % 5 == 0 or epoch == NUM_EPOCHS:
            print(f"        Ep{epoch:02d}  tr={tr_acc:.4f}  vl={vl_acc:.4f}  best={best_val_acc:.4f}")

        if no_improve >= PATIENCE:
            print(f"        ⏹ Early stop ep{epoch}")
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, _, vl_preds, vl_labels = eval_epoch(model, vl_loader, criterion, model_name)
    n_tta = 8 if model_name == "DiaRetULS-Net" else 4
    test_logits = predict_tta(model, test_df, model_name, n_tta=n_tta)

    del optimizer, scheduler, tr_loader, vl_loader, best_state
    torch.cuda.empty_cache(); gc.collect()
    return model, vl_preds, vl_labels, test_logits


def train_diaretuls_run(train_df, val_df, test_df, dataset_name, run_id):
    """DiaRetULS-Net: 5-Fold × 3 Seeds ensemble per run."""
    run_metrics     = []
    all_test_logits = []
    criterion       = HybridLoss()
    full_train      = pd.concat([train_df, val_df]).reset_index(drop=True)
    skf             = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=run_id)

    for fold, (tr_idx, vl_idx) in enumerate(
            skf.split(full_train, full_train["label"]), start=1):
        fold_tr = full_train.iloc[tr_idx].reset_index(drop=True)
        fold_vl = full_train.iloc[vl_idx].reset_index(drop=True)
        print(f"    Fold {fold}/{NUM_FOLDS}  train={len(fold_tr)} val={len(fold_vl)}")

        fold_logits = []
        for seed in range(NUM_SEEDS):
            print(f"      Seed {seed} ...")
            model, vl_preds, vl_labels, test_logits = _train_one_model(
                fold_tr, fold_vl, test_df, seed, run_id, fold,
                criterion, "DiaRetULS-Net")
            m = compute_metrics(vl_labels, vl_preds, "DiaRetULS-Net",
                                dataset_name, run_id, fold, seed)
            run_metrics.append(m)
            fold_logits.append(test_logits)
            del model; torch.cuda.empty_cache(); gc.collect()

        all_test_logits.append(torch.stack(fold_logits).mean(0))

    # Ensemble over all folds × seeds
    ens_logits = torch.stack(all_test_logits).mean(0)
    ens_preds  = ens_logits.argmax(1).numpy()
    m_test = compute_metrics(test_df["label"].values, ens_preds,
                             "DiaRetULS-Net", dataset_name, run_id, "ensemble", "all")
    run_metrics.append(m_test)
    print(f"    ✅ Run {run_id} ensemble → acc={m_test['accuracy']:.4f}  f1={m_test['f1_score']:.4f}")
    return run_metrics


def train_baseline_run(train_df, val_df, test_df, dataset_name, run_id, model_name):
    """Baseline model: single-split training per run."""
    criterion = get_criterion(model_name)
    model, vl_preds, vl_labels, test_logits = _train_one_model(
        train_df, val_df, test_df, run_id * 7, run_id, 1, criterion, model_name)

    m_val  = compute_metrics(vl_labels, vl_preds, model_name,
                              dataset_name, run_id, "val", "single")
    ens_preds = test_logits.argmax(1).numpy()
    m_test = compute_metrics(test_df["label"].values, ens_preds,
                              model_name, dataset_name, run_id, "ensemble", "all")
    del model; torch.cuda.empty_cache(); gc.collect()
    print(f"    ✅ Run {run_id} → acc={m_test['accuracy']:.4f}  f1={m_test['f1_score']:.4f}")
    return [m_val, m_test]


print("✅ CELL 10 complete — Training engine ready")


In [ ]:
# ============================================================
# CELL 11 — MAIN TRAINING LOOP
# Runs all models × all datasets × 20 runs
# Saves per-model CSV and final comparison CSV
# ============================================================

DATASETS_TO_RUN = ["Messidor2", "APTOS2019", "IDRiD"]
MODELS_TO_RUN   = MODEL_NAMES   # All 7 models
# To run only specific models/datasets, edit the lists above.
# e.g. MODELS_TO_RUN = ["DiaRetULS-Net", "VGG19"]

# Final summary collector
all_averages = []

for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*65}")
    print(f"  DATASET : {dataset_name}")
    print(f"{'='*65}")

    try:
        loader_fn = LOADERS[dataset_name]
        root      = DATASET_PATHS[dataset_name]
        train_df, val_df, test_df = loader_fn(root)
        print(f"  Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")
    except Exception as e:
        print(f"  ❌ Could not load {dataset_name}: {e}")
        continue

    for model_name in MODELS_TO_RUN:
        print(f"\n  ── Model: {model_name} ──────────────────────────")
        csv_path = get_csv_path(model_name, dataset_name)

        # Count already completed runs (resume support)
        done_runs = 0
        if os.path.exists(csv_path):
            done_df   = pd.read_csv(csv_path)
            done_runs = done_df[done_df["fold"] == "ensemble"]["run"].nunique()
            print(f"     Resuming from run {done_runs+1}/{NUM_RUNS}")

        for run_id in range(done_runs + 1, NUM_RUNS + 1):
            t0 = time.time()
            print(f"\n  ▶ Run {run_id}/{NUM_RUNS}  [{model_name} | {dataset_name}]")

            try:
                if model_name == "DiaRetULS-Net":
                    metrics = train_diaretuls_run(
                        train_df, val_df, test_df, dataset_name, run_id)
                else:
                    metrics = train_baseline_run(
                        train_df, val_df, test_df, dataset_name, run_id, model_name)

                append_to_csv(metrics, model_name, dataset_name)
                elapsed = (time.time() - t0) / 60
                print(f"  ✔ Run {run_id} saved → {os.path.basename(csv_path)}  ({elapsed:.1f} min)")

            except Exception as exc:
                import traceback; traceback.print_exc()
                print(f"  ❌ Run {run_id} failed: {exc}")
                continue

        # Compute average after all runs
        if os.path.exists(csv_path):
            avg = compute_average_metrics(csv_path, model_name, dataset_name)
            all_averages.append(avg)
            print(f"\n  📊 {model_name} | {dataset_name} — 20-Run Average:")
            print(f"     Acc={avg['accuracy']}%  Prec={avg['precision']}%  "
                  f"Sens={avg['sensitivity']}%  Spec={avg['specificity']}%  "
                  f"F1={avg['f1_score']}%")

print("\n✅ CELL 11 complete — All training done")


In [ ]:
# ============================================================
# CELL 12 — VGG19 CONSOLIDATED CSV (All 3 Datasets)
# Creates: VGG19_all_datasets_20runs.csv
# Columns : dataset, run, fold, seed, accuracy, precision,
#            sensitivity, specificity, f1_score
# ============================================================

import pandas as pd, os

vgg19_dfs = []
for ds in ["Messidor2", "APTOS2019", "IDRiD"]:
    p = get_csv_path("VGG19", ds)
    if os.path.exists(p):
        df_part = pd.read_csv(p)
        vgg19_dfs.append(df_part)
        print(f"  VGG19 | {ds}: {len(df_part)} rows")
    else:
        print(f"  ⚠️  VGG19 | {ds}: CSV not found ({p})")

if vgg19_dfs:
    vgg19_all = pd.concat(vgg19_dfs, ignore_index=True)
    out_path  = os.path.join(OUTPUT_DIR, "VGG19_all_datasets_20runs.csv")
    vgg19_all.to_csv(out_path, index=False)
    print(f"\n✅ VGG19 consolidated CSV saved → {out_path}")
    print(f"   Shape: {vgg19_all.shape}")
    # Per-dataset summary
    print("\n  VGG19 — 20-Run Averages (ensemble rows only):")
    ens = vgg19_all[vgg19_all["fold"] == "ensemble"]
    for ds in ["Messidor2", "APTOS2019", "IDRiD"]:
        sub = ens[ens["dataset"] == ds]
        if len(sub):
            print(f"  {ds:12s}  Acc={sub['accuracy'].mean()*100:.2f}%  "
                  f"Prec={sub['precision'].mean()*100:.2f}%  "
                  f"Sens={sub['sensitivity'].mean()*100:.2f}%  "
                  f"Spec={sub['specificity'].mean()*100:.2f}%  "
                  f"F1={sub['f1_score'].mean()*100:.2f}%")
else:
    print("  No VGG19 CSVs found. Run CELL 11 first.")


In [ ]:
# ============================================================
# CELL 13 — COMPARATIVE PERFORMANCE TABLE & VISUALISATION
# Generates the paper-quality comparison table + bar charts
# ============================================================

import pandas as pd, os, matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Load all results ─────────────────────────────────────────
records = []
for model_name in MODEL_NAMES:
    for ds in ["Messidor2", "APTOS2019", "IDRiD"]:
        p = get_csv_path(model_name, ds)
        if os.path.exists(p):
            avg = compute_average_metrics(p, model_name, ds)
            records.append(avg)
        else:
            print(f"  ⚠️  Missing: {model_name} | {ds}")

if not records:
    print("No results found. Run CELL 11 first.")
else:
    df_comp = pd.DataFrame(records)
    df_comp = df_comp.sort_values(["dataset","accuracy"], ascending=[True,False])

    # ── Pretty print ─────────────────────────────────────────
    print("\n" + "="*90)
    print("  COMPARATIVE PERFORMANCE — DiaRetULS-Net vs Baselines")
    print("="*90)
    print(df_comp.to_string(index=False))

    # ── Save comparison CSV ───────────────────────────────────
    comp_csv = os.path.join(OUTPUT_DIR, "Comparative_Performance_All_Models.csv")
    df_comp.to_csv(comp_csv, index=False)
    print(f"\n📁 Comparison CSV saved → {comp_csv}")

    # ── Bar chart: Accuracy by model for each dataset ─────────
    COLORS = {
        "DiaRetULS-Net": "#7B2D8B",
        "VGG19":         "#2196F3",
        "LSTM":          "#FF9800",
        "AlexNet":       "#F44336",
        "InceptionV3":   "#4CAF50",
        "ResNet50":      "#00BCD4",
        "DenseNet121":   "#795548",
    }

    for metric, ylabel in [("accuracy","Accuracy (%)"), ("f1_score","F1-Score (%)")]:
        fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=False)
        fig.suptitle(f"Comparative {ylabel} — DiaRetULS-Net vs Baselines",
                     fontsize=16, fontweight="bold", y=1.02)

        for ax, ds in zip(axes, ["Messidor2","APTOS2019","IDRiD"]):
            sub = df_comp[df_comp["dataset"] == ds].sort_values(metric, ascending=False)
            bars = ax.bar(sub["model"], sub[metric],
                          color=[COLORS.get(m,"#888") for m in sub["model"]],
                          edgecolor="white", linewidth=0.8)
            ax.set_title(ds, fontsize=13, fontweight="bold")
            ax.set_xlabel("Model", fontsize=11)
            ax.set_ylabel(ylabel, fontsize=11)
            ax.set_ylim(max(0, sub[metric].min() - 3), min(100, sub[metric].max() + 2))
            ax.tick_params(axis="x", rotation=35)
            ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f"))
            for bar, val in zip(bars, sub[metric]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                        f"{val:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
            ax.grid(axis="y", alpha=0.3, linestyle="--")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        plt.tight_layout()
        out_fig = os.path.join(OUTPUT_DIR, f"Comparison_{metric}.png")
        plt.savefig(out_fig, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"📊 Chart saved → {out_fig}")

    # ── Heatmap ───────────────────────────────────────────────
    for ds in ["Messidor2","APTOS2019","IDRiD"]:
        sub = df_comp[df_comp["dataset"]==ds].set_index("model")
        metrics_cols = ["accuracy","precision","sensitivity","specificity","f1_score"]
        sub_m = sub[metrics_cols].copy()
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.heatmap(sub_m.T, annot=True, fmt=".2f", cmap="YlOrRd",
                    linewidths=0.5, ax=ax, vmin=85, vmax=100)
        ax.set_title(f"{ds} — All Metrics Heatmap", fontsize=14, fontweight="bold")
        plt.tight_layout()
        hm_path = os.path.join(OUTPUT_DIR, f"Heatmap_{ds}.png")
        plt.savefig(hm_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"📊 Heatmap saved → {hm_path}")

print("\n✅ CELL 13 complete")


In [ ]:
# ============================================================
# CELL 14 — HOW TO RUN (VS Code & GPU Server)
# ============================================================

instructions = """
╔══════════════════════════════════════════════════════════════════════╗
║           DiaRetULS-Net — Full Pipeline Run Instructions            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  ── LOCAL VS CODE ─────────────────────────────────────────────     ║
║  1. Install extensions: Python, Jupyter                              ║
║  2. Open this .ipynb in VS Code                                      ║
║  3. Select your Python/CUDA kernel (top-right of notebook)           ║
║  4. Run cells in order: 1 → 2 → 3 → ... → 14                        ║
║                                                                      ║
║  ── GPU SERVER (recommended) ───────────────────────────────────     ║
║  1. Push this notebook to GitHub:                                    ║
║       git add DiaRetULS_Full_Pipeline.ipynb                          ║
║       git commit -m "full pipeline"                                  ║
║       git push origin main                                           ║
║                                                                      ║
║  2. On GPU server (SSH):                                             ║
║       git clone https://github.com/YOUR_USER/YOUR_REPO.git           ║
║       cd YOUR_REPO                                                   ║
║                                                                      ║
║  3. Set Kaggle credentials:                                          ║
║       export KAGGLE_USERNAME="your_username"                         ║
║       export KAGGLE_KEY="your_api_key"                               ║
║     OR copy kaggle.json to ~/.kaggle/kaggle.json                     ║
║                                                                      ║
║  4. Run notebook headlessly (saves all outputs to /results/):        ║
║       pip install jupyter nbconvert -q                               ║
║       jupyter nbconvert --to notebook --execute                      ║
║           --inplace DiaRetULS_Full_Pipeline.ipynb                    ║
║           --ExecutePreprocessor.timeout=999999                       ║
║                                                                      ║
║  5. Or run with screen/tmux (survives SSH disconnect):               ║
║       tmux new -s dr_train                                           ║
║       jupyter nbconvert --execute --inplace DiaRetULS_Full_Pipeline.ipynb ║
║       [Ctrl+B then D to detach]                                      ║
║                                                                      ║
║  ── DATASETS (Kaggle API auto-downloads) ────────────────────────    ║
║  • APTOS 2019 : aptos2019-blindness-detection (Kaggle competition)   ║
║  • IDRiD      : andrewmvd/indian-diabetic-retinopathy-image-dataset  ║
║  • Messidor-2 : sovitrath/diabetic-retinopathy-224x224-2019-data     ║
║                                                                      ║
║  ── OUTPUT FILES ────────────────────────────────────────────────    ║
║  results/                                                            ║
║  ├── DiaRetULS-Net_Messidor2_20runs_metrics.csv                      ║
║  ├── DiaRetULS-Net_APTOS2019_20runs_metrics.csv                      ║
║  ├── DiaRetULS-Net_IDRiD_20runs_metrics.csv                          ║
║  ├── VGG19_Messidor2_20runs_metrics.csv                              ║
║  ├── VGG19_APTOS2019_20runs_metrics.csv                              ║
║  ├── VGG19_IDRiD_20runs_metrics.csv                                  ║
║  ├── VGG19_all_datasets_20runs.csv    ← VGG19 consolidated           ║
║  ├── ... (one CSV per model×dataset)                                 ║
║  ├── Comparative_Performance_All_Models.csv                          ║
║  ├── Comparison_accuracy.png                                         ║
║  ├── Comparison_f1_score.png                                         ║
║  └── Heatmap_Messidor2.png / Heatmap_APTOS2019.png / ...            ║
║                                                                      ║
║  ── RESUMING INTERRUPTED RUNS ──────────────────────────────────     ║
║  The training loop (CELL 11) auto-resumes from where it stopped.     ║
║  Just re-run the cell — it counts completed ensemble rows in CSV.    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(instructions)
